# Headline sentiment

Associated chart: the headline table and the framing chart's tone axis

Output files:
`headline_sentiment.csv` and `sentiment_outlets.csv` in `src/data/`.

Input files: `data/fox_wnba.csv`, `data/all_outlets_index.csv`.

Requirements: torch + transformers (~10 minutes on an Apple Silicon Mac)

`pip install torch transformers`


## Setup

In [1]:
import json, re
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DATA = Path("../data")
OUT = Path("../src/data")

MODEL = "cardiffnlp/twitter-roberta-large-topic-sentiment-latest"
BATCH = 64
# the score is a weighted average of the model's five classes,
# from strongly negative (-1) to strongly positive (+1)
LABEL_WEIGHTS = [("strongly negative", -1.0), ("strongly positive", 1.0),
                 ("negative or neutral", 0.0), ("positive or neutral", 0.0),
                 ("negative", -0.6), ("positive", 0.6), ("neutral", 0.0)]
BUCKET_EDGE = 0.15

def weight_for(label):
    return next((w for phrase, w in LABEL_WEIGHTS if phrase in label), 0.0)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device).eval()
weights = [weight_for(model.config.id2label[i].lower()) for i in range(model.config.num_labels)]

def score(texts):
    out = []
    for start in range(0, len(texts), BATCH):
        chunk = texts[start:start + BATCH]
        encoded = tokenizer(chunk, padding=True, truncation=True, max_length=64,
                            return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.softmax(model(**encoded).logits, dim=-1).cpu().tolist()
        out += [round(sum(p * w for p, w in zip(prob, weights)), 4) for prob in probs]
        if (start // BATCH) % 40 == 0:
            print(f"  {start + len(chunk):,}/{len(texts):,}", flush=True)
    return out

print(f"model on {device}")

/Users/melwalsh/.local/share/uv/python/cpython-3.13.15-macos-aarch64-none/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model on mps


## Every Fox headline


In [2]:
fox = pd.read_csv(DATA / "fox_wnba.csv")
fox_headlines = list(dict.fromkeys(h.strip() for h in fox.headline.dropna() if h.strip()))
fox_scores = score(fox_headlines)

def bucket(s):
    return "negative" if s <= -BUCKET_EDGE else "positive" if s >= BUCKET_EDGE else "neutral"

sent = (pd.DataFrame({"headline": fox_headlines, "score": fox_scores})
          .assign(label=lambda d: d.score.map(bucket))
          .sort_values("score"))

  64/4,351


  2,624/4,351


#### Overwrites `src/data/headline_sentiment.csv`

In [3]:
sent.to_csv(OUT / "headline_sentiment.csv", index=False)
print(sent.label.value_counts().to_dict())
sent.head(5)

{'neutral': 2837, 'positive': 886, 'negative': 628}


,headline,score,label
3295,Indiana Fever Fans Have WNBA Player On Verge O...,-0.6529,negative
2475,Natasha Cloud Sends Insanely Stupid Anti-Donal...,-0.6383,negative
3439,WNBA Officiating Is Embarrassing And It's Hija...,-0.6185,negative
653,Kamala Harris is a Hypocrite on Brittney Grine...,-0.6038,negative
3937,Nancy Armour's disastrous Caitlin Clark articl...,-0.5979,negative


## Every outlet, per player


In [4]:
index = pd.read_csv(DATA / "all_outlets_index.csv")
others = (index[~index.outlet.str.startswith("Fox")]
          .assign(headline=lambda d: d.headline.str.strip())
          .dropna(subset=["headline"])
          .drop_duplicates(["outlet", "headline"]))
other_scores = dict(zip(zip(others.outlet, others.headline), score(list(others.headline))))

players = {name for name, p in json.loads((OUT / "people.json").read_text()).items()
           if p["group"] == "players"}
# capitalized surnames only: "Brink" the player, never "the brink of suspension"
name_rx = {n: re.compile(r"\b" + re.escape(n.split()[-1]) + r"\b") for n in players}

pairs = defaultdict(list)
for (outlet, headline), s in other_scores.items():
    for name, rx in name_rx.items():
        if rx.search(headline):
            pairs[(name, outlet)].append(s)
for headline, s in zip(fox_headlines, fox_scores):
    for name, rx in name_rx.items():
        if rx.search(headline):
            pairs[(name, "Fox News")].append(s)

rows = [{"player": name, "outlet": outlet, "n": len(v), "mean": round(sum(v) / len(v), 4)}
        for (name, outlet), v in sorted(pairs.items()) if len(v) >= 12]

  64/43,746


  2,624/43,746


  5,184/43,746


  7,744/43,746


  10,304/43,746


...

  33,344/43,746


  35,904/43,746


  38,464/43,746


  41,024/43,746


  43,584/43,746


#### Overwrites `src/data/sentiment_outlets.csv`

In [5]:
pd.DataFrame(rows).to_csv(OUT / "sentiment_outlets.csv", index=False)
pd.DataFrame(rows).query("outlet in ('Fox News', 'Yahoo Sports')").sort_values(["player", "outlet"]).head(12)

,player,outlet,n,mean
2,A'ja Wilson,Fox News,64,0.0145
6,A'ja Wilson,Yahoo Sports,690,0.1690
9,Aliyah Boston,Fox News,27,0.1666
13,Aliyah Boston,Yahoo Sports,704,0.0883
16,Alyssa Thomas,Fox News,24,0.0033
20,Alyssa Thomas,Yahoo Sports,210,-0.0015
23,Angel Reese,Fox News,589,0.0037
27,Angel Reese,Yahoo Sports,3698,0.0976
32,Arike Ogunbowale,Yahoo Sports,45,0.0388
35,Breanna Stewart,Fox News,30,0.1305


## Spot check

In [6]:
import random

print("— random Fox headlines with their scores")
for _, r in sent.sample(6).sort_values("score").iterrows():
    print(f"   {r.score:+.2f} {r.label:8s} {r.headline[:80]}")

print("\n— random other-outlet headlines")
for (outlet, headline), s in random.sample(list(other_scores.items()), 4):
    print(f"   {s:+.2f} [{outlet}] {headline[:74]}")

— random Fox headlines with their scores
   -0.11 neutral  Angel Reese Receiving A 4th Place WNBA MVP Vote Is Categorically Insane
   -0.05 neutral  Texas lawmaker says WNBA response on Caitlin Clark ‘falls short,’ keeps ‘every o
   +0.04 neutral  Caitlin Clark Props For WNBA Debut Tuesday Are The Hottest Bets In Sports
   +0.20 positive Alex Rodriguez's daughter sings national anthem before WNBA game
   +0.54 positive Angel Reese ecstatic as her brother Julian signs with Lakers after going undraft
   +0.54 positive Golf legend Annika Sorenstam speaks glowingly of Caitlin Clark after teaming up 

— random other-outlet headlines
   +0.11 [Just Women's Sports] Breanna Stewart plans to keep pushing WNBA for better flights
   +0.16 [Yahoo Sports] WNBA players to earn millions for 1st time in revenue share
   +0.33 [Yahoo Sports] Angel Reese Challenges WNBA MVP A'ja Wilson in Thrilling 85-84 Game
   +0.01 [Yahoo Sports] Angel Reese Makes Career Decision After Second WNBA Season
